In [ ]:
from config import get_config
from main import run_pipeline
from utils.experiment_logger import log_experiment

models = ["ANN",'RF','XGB']
targets = ['L']
stabilities = [ 'stable','unstable','all']
data = ['IRG']
from itertools import product
experiments = list(product(models, targets, stabilities,data))

print("Total experiments:", experiments)


for model, target, stability,data in experiments:

    print("\n" + "="*60)
    print(f"Running: {model} | {target} | {stability} |{data}")
    print("="*60)

    # -------- TRAIN --------
    config = get_config(
        model=model,
        target=target,
        stability=stability,
        data=data,
        mode="train"
    )

    run_pipeline(config)

    # -------- PREDICT --------
    config.mode = "predict"

    metrics,pred,obs = run_pipeline(config)

    # -------- LOG --------
    log_experiment(config, metrics,extra_params={"type":"After HPT but H.Tau,s and U hyp para same as csat"})

### Ensemble evaluation
Model is dependent on how the weights are biases are initialised in ANN, how the samples are fed to the trees in RF, and colsampling for XGB
So we run the models for 100 different seeds, and average the predicitons as well as the metrics.
For the final model to be deployed, we can not take average of weights, we either will pick the best seed and train the model using the whole data.
For seed ensembling, we will keep the train test split constant for all the models and the ensemble.

In [ ]:
from config import get_config
from main import run_pipeline
from utils.experiment_logger import log_experiment
import os
from itertools import product
import numpy as np
import pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

targets = ["H","TAU","L"]
models = ["ANN", "RF", "XGB"]
stabilities = ["unstable","stable"]
experiments = list(product(models, targets, stabilities))
valid_ranges = {"H": (-100, 600),"TAU": (0, 3),"L": (-500, 500)}

split_seed = 42              # FIXED DATA SPLIT
model_seeds = [s*5 for s in range(1,101)]  # MODEL VARIABILITY

os.makedirs("outputs/ensemble_preds/Ensemble_irg", exist_ok=True)
os.makedirs("outputs/seed_metrics/Ensemble_irg", exist_ok=True)

all_results = []

for model, target, stability in experiments:

    print("\n" + "="*70)
    print(f"RUNNING ENSEMBLE: {model} | {target} | {stability}")

    preds_all = []
    metrics_all = []
    seed_records=[]
    for seed in model_seeds:
        config = get_config(
            model=model,
            target=target,
            stability=stability,
            mode="train",
            split_seed=split_seed,
            model_seed=seed)

        run_pipeline(config)
        config.mode = "predict"
        metrics, y_pred, y_test = run_pipeline(config)
        preds_all.append(y_pred)
        metrics_all.append(metrics)
        seed_records.append({"model": model,"target": target,"stability": stability,
                                "seed": seed,"RMSE": metrics["rmse"],"MAE": metrics["mae"],
                                "R2": metrics["r2"],  "Correlation":metrics["corr"]})

    preds_all = np.array(preds_all)
    ymin, ymax = valid_ranges[target]
    valid = (preds_all >= ymin) & (preds_all <= ymax)
    n_valid = np.sum(valid, axis=0)
    sum_valid = np.sum(np.where(valid, preds_all, 0), axis=0)
    y_pred_mean = sum_valid / n_valid
    
    # y_pred_mean = np.mean(preds_all, axis=0)
    y_true = y_test  # same split
    rmse = np.sqrt(mean_squared_error(y_true, y_pred_mean))
    mae = mean_absolute_error(y_true, y_pred_mean)
    r2 = r2_score(y_true, y_pred_mean)
    corr = np.corrcoef(y_true, y_pred_mean)[0, 1]

    ensemble_metrics = {"RMSE": rmse,"MAE": mae,"R2": r2, "CORR":corr}
    # print("\n ENSEMBLE RESULTS:", ensemble_metrics)

    # -------------------------------
    # SEED STATISTICS
    # -------------------------------
    rmse_list = [m["rmse"] for m in metrics_all]
    mae_list = [m["mae"] for m in metrics_all]
    r2_list = [m["r2"] for m in metrics_all]

    stats = {"RMSE_mean": np.mean(rmse_list),"RMSE_std": np.std(rmse_list),
            "MAE_mean": np.mean(mae_list),"MAE_std": np.std(mae_list),
            "R2_mean": np.mean(r2_list),"R2_std": np.std(r2_list),}


    # -------------------------------
    # SAVE ENSEMBLE PREDICTIONS
    # -------------------------------
    df_pred = pd.DataFrame({"y_true": y_true,"y_pred": y_pred_mean})
    pred_path = f"outputs/ensemble_preds/Ensemble_irg/{model}_{target}_{stability}.csv"
    df_pred.to_csv(pred_path, index=False)
    
    df_seed = pd.DataFrame(seed_records)
    file_path = f"outputs/seed_metrics/Ensemble_irg/{model}_{target}_{stability}.csv"
    df_seed.to_csv(file_path, index=False)
    # -------------------------------
    # LOG RESULTS
    # -------------------------------
    log_config = get_config(model=model,target=target,stability=stability,mode="predict",split_seed=split_seed,model_seed=-1 )
    log_experiment(log_config,ensemble_metrics,extra_params={**stats,"type": "ensemble_irg"})

    # Store for summary table
    all_results.append({"model": model,"target": target,"stability": stability,**ensemble_metrics,**stats})

results_df = pd.DataFrame(all_results)
print("\n FINAL RESULTS")
print(results_df)

results_df.to_csv("outputs/final_summary_ens_irg.csv", index=False)
print("\n[Saved] outputs/final_summary_irg.csv")

In [ ]:
def apply_physical_filter(y_true, y_pred, target, stability):

    import numpy as np

    y_true = np.array(y_true)
    y_pred = np.array(y_pred)

    if target == "H":
        if stability == "stable":
            mask = y_true > -100
        else:  # unstable
            mask = y_true < 500

    elif target == "TAU":
        mask = np.abs(y_true) < 3

    else:
        mask = np.ones_like(y_true, dtype=bool)

    return y_true[mask], y_pred[mask]

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
df = pd.read_excel('~/IIT_Roorkee/Work/arctic/Ranichauri/data/irgasen_ranichauri.xlsx', engine='openpyxl')

df_clean = df[
    (df['H_QC'].between(1, 6)) &    
    (df['LE_QC'].between(1, 6))&# QC filter
    (df['H_kin'].between(-500, 500)) &               # X range
    (df['H_sens'].between(-500, 500))                # Y range
]
    #[['H_kin', 'H_sens']].dropna()                      # remove NaNs

x = df_clean['H_kin']
y = df_clean['H_sens']

# Fit line: y = m*x + c
m, c = np.polyfit(x, y, 1)

# Plot scatter
plt.plot(x, y, '.', label='Data')

# Plot fitted line
plt.plot(x, m*x + c, 'r-', label=f'Fit: y={m:.2f}x + {c:.2f}')

plt.xlabel('H_kin')
plt.ylabel('H_sens')
plt.legend()
plt.grid(True)

print(f"Slope (m): {m}")
print(f"Intercept (c): {c}")

plt.show()
r = np.corrcoef(x, y)[0, 1]
print(f"Correlation (r): {r}")
n = len(x)
se_r = np.sqrt((1 - r**2) / (n - 2))

print(f"Standard Error of r: {se_r}")